In [240]:
import altair as alt
import pandas as pd

In [241]:
from pathlib import Path
DATA_DIR = Path().resolve().parent / "data"

combined_data_df = pd.read_csv(DATA_DIR / "cleaned/all_data_combined.csv")

combined_data_df["Years colonized"] = (
    combined_data_df.groupby("Entity")["Years colonized"].transform("max")
)

In [242]:
# Add country codes to the df for choropleths
import pycountry

def to_numeric(code):
    try:
        return int(pycountry.countries.get(alpha_3=code).numeric)
    except Exception:
        return None

combined_data_df["id"] = combined_data_df["Code"].map(to_numeric)

In [243]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

### Legal Status of Homosexuality Choropleth

In [271]:
def world_base():
    return (
        alt.Chart(countries)
        .mark_geoshape(fill="lightgray")
    )

In [277]:
legal_status_colors = {
    "Legal": "#e6d921",
    "Illegal": "#ff8d00",
    "No data" : "lightgray"
}

In [281]:
from vega_datasets import data

countries = alt.topo_feature(data.world_110m.url, "countries")

def legal_status_choropleth(df: pd.DataFrame, year: str):
    choropleth = (
        alt.Chart(countries)
        .mark_geoshape(stroke="black", strokeWidth=0.25)
        .transform_lookup(
            lookup="id",
            from_=alt.LookupData(df, "id", ["Entity", "Legal status of homosexuality"]),
        )
        .encode(
            color=alt.Color(
                "Legal status of homosexuality:N",
                title="Legal Status",
                scale=alt.Scale(
                    domain=legal_status_colors.keys(),
                    range=legal_status_colors.values()
                )
                ),
        )
    )

    return (
        world_base() + choropleth
        .properties(title=year, width=600, height=300)
        .project(type="equalEarth")  
    )

In [ ]:
df_1970 = combined_data_df[combined_data_df["Year"] == 1970]
df_2025 = combined_data_df[combined_data_df["Year"] == 2025]

choropleth_1970 = legal_status_choropleth(df_1970, "1970")
choropleth_2025 = legal_status_choropleth(df_2025, "2025")

multiple_charts = choropleth_1970 & choropleth_2025
multiple_charts.properties(title="Legal Status of Same-Sex Sexual Acts")

### Longest colonizer bar chart

In [248]:
# Equaldex dataframe
equaldex_df = pd.read_csv(DATA_DIR/"cleaned/equaldex_with_colonial_history.csv")
equaldex_df = equaldex_df.dropna(subset=["EI", "EI Legal", "EI PO", "Longest colonizer", "Continent"])

In [ ]:
bars = (
    alt.Chart(equaldex_df)
    .mark_bar()
    .encode(
        x=alt.X("mean(EI):Q", title="Mean Equality Index"),
        y=alt.Y("Longest colonizer:N", sort="-x"),
        color=alt.value("#760188"),
        tooltip=["Longest colonizer:N", "mean(EI):Q", "count():Q"],
    )
    .properties(width=650, height=350, title="Mean Equality Index of countries with the same longest colonizer")
)

bars

### Scatter plots (by continent and colonization status)

In [284]:
continent_color_scheme = {
    "Africa": "#760188",
    "Asia": "#004cff",
    "Europe": "#008121",
    "Oceania": "#e6d921",
    "North America": "#ff8d00",
    "South America": "#e50000",
}

In [285]:
def equality_index_scatter_plot(df, colonization_status):
    scatter = (
        alt.Chart(df)
        .mark_circle(size=70, opacity=0.7)
        .encode(
            x=alt.X("EI Legal:Q", title="Legal Equality Index"),
            y=alt.Y("EI PO:Q", title="Public Opinion Equality Index", scale=alt.Scale(domain=[0, 85])),
            color=alt.Color(
                "Continent:N",
                scale=alt.Scale(
                    domain=continent_color_scheme.keys(),
                    range=continent_color_scheme.values()
                )
            ),
            tooltip=["Name:N", "EI:Q", "EI Legal:Q", "EI PO:Q", "Years colonized:Q", "Longest colonizer:N"],
        )
        .properties(title=colonization_status)
    )

    return scatter

In [ ]:
colonized_only = equaldex_df[equaldex_df["Longest colonizer"] != "Not colonized"]
colonized_continent_scatter = equality_index_scatter_plot(colonized_only, "Colonized")

not_colonized = equaldex_df[equaldex_df["Longest colonizer"] == "Not colonized"]
not_colonized_continent_scatter = equality_index_scatter_plot(not_colonized, "Not colonized")

(colonized_continent_scatter | not_colonized_continent_scatter).properties(title="Equality Index by Continent and Colonization")

### Box plots

In [287]:
equality_index_box_plot = (
    alt.Chart(equaldex_df)
    .mark_boxplot()
    .encode(
        x="Continent:N", 
        y=alt.Y("EI:Q", title="Equality Index"), 
        color=alt.Color(
            "Continent:N",
            scale=alt.Scale(
                domain=continent_color_scheme.keys(),
                range=continent_color_scheme.values()
            )
        )
    )
    .properties(title="LGBT Equality Index")
)

In [288]:
years_colonized_box_plot = (
    alt.Chart(equaldex_df)
    .mark_boxplot()
    .encode(
        x="Continent:N", 
        y=alt.Y("mean(Years colonized):Q", title="Mean years colonized"), 
        color=alt.Color(
            "Continent:N",
            scale=alt.Scale(
                domain=continent_color_scheme.keys(),
                range=continent_color_scheme.values()
            )
        )
    )
    .properties(title="Mean Years Colonized")
)

In [ ]:
equality_index_box_plot | years_colonized_box_plot